# Image segmentation with Keras

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy
import pandas
import tensorflow as tf
from tensorflow import keras
from tifffile import imread

In [ ]:
!nvidia-smi

In [ ]:
%%time
!rm -rf sample_data
!git clone https://github.com/mlambda/ign-FNF-2009.git

## Data

The data are aerial photographs taken by IGN (the French national mapping agency) in 2009 in Lorraine, in north-eastern France: 20 areas of 5 km × 5 km. Each area is cut into 9 × 9 = 81 tiles of 1024 × 1024 pixels (1 pixel = 50 cm on the ground: a tile covers 512 m × 512 m).

- **Input** (`*_irc.tif` files): a *color infrared* (CIR) image. Its three channels are near infrared, red and green, displayed as red, green and blue. Vegetation strongly reflects infrared: it shows up in red.
- **Target** (`*_pfnf_nch.tif` files): the class of each pixel, forest or non-forest, one-hot encoded over two channels. The code keeps its index: 1 for forest, 0 for non-forest. In IGN's sense, a forest isn't just a group of trees: it's an area of more than 5,000 m², at least 20 m wide, where vegetation covers at least 10% of the ground and which isn't used for farming.

2 areas were set aside for validation, and 2 others for testing. All the tiles of an area go to the same set: neighboring tiles look too much alike to measure generalization honestly.

In [ ]:
df_train = pandas.read_csv("ign-FNF-2009/train-files.csv")
df_val = pandas.read_csv("ign-FNF-2009/val-files.csv")
df_test = pandas.read_csv("ign-FNF-2009/test-files.csv")

print("Training coordinates")
print(df_train.coord.unique())

print("Validation coordinates")
print(df_val.coord.unique())

print("Test coordinates")
print(df_test.coord.unique())

df_test

## Study of an example and its annotation

In [ ]:
example_input = imread(df_train.Observations[0])
example_annotation = imread(df_train.Annotations[0])

In [ ]:
def display_sample(*images: numpy.array) -> None:
  """Show side-by-side an input image, the ground truth and the prediction."""
  figure, axes = plt.subplots(1, len(images), figsize=(18, 18))

  titles = ["Input", "Target", "Prediction"]
  for image, ax, title in zip(images, axes, titles):
    ax.set_title(title)
    if image.ndim == 3 and image.shape[2] == 3:
      # Color infrared image
      ax.imshow(image)
    else:
      if image.ndim == 3:
        # Channel of the forest class (the last one)
        image = image[..., -1]
      # Fixed bounds: forest (1) in yellow, non-forest (0) in purple, even when
      # the whole tile is of a single class
      ax.imshow(image, vmin=0, vmax=1)
    ax.axis("off")
  plt.show()


display_sample(example_input, example_annotation)

## General options

In [ ]:
# Original image size
dim_original = 1024

# Input size
dim_input = 512

# Number of image channel
num_channel = 3

## Data augmentation

In image processing, it is extremely important to augment the data with transforms to make the learned network less sensitive to rotations, cropping, etc. Implement the `transform_data` function, which, if `eval` is `True` will simply do a centred crop of the `X` and `Y` images, and otherwise do whatever transformations you deem appropriate on these same arrays for training (random cropping, mirroring, rotation, colour noise, brightness, etc).

To do this, use the functions in the [`tf.image`] module (https://www.tensorflow.org/api_docs/python/tf/image/).

In [ ]:
def transform_data(
  X: numpy.ndarray,
  Y: numpy.ndarray,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[numpy.ndarray, numpy.ndarray]:
  # Votre code ici
  return X, Y


# display_sample(*transform_data(example_input, example_annotation))

### Solution

In [ ]:
def transform_data(
  X: numpy.ndarray,
  Y: numpy.ndarray,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[numpy.ndarray, numpy.ndarray]:
  height, width = X.shape[:2]
  cropped_height, cropped_width = cropped_size
  if eval:
    # Centered cropping
    crop_params = dict(
      offset_height=(height - cropped_height) // 2,
      offset_width=(width - cropped_width) // 2,
      target_height=cropped_height,
      target_width=cropped_width,
    )
    X = tf.image.crop_to_bounding_box(X, **crop_params)
    Y = tf.image.crop_to_bounding_box(Y, **crop_params)
  else:
    # Random Rotation
    k = random.choice([None, 1, 3])
    if k is not None:
      X = tf.image.rot90(X, k)
      Y = tf.image.rot90(Y, k)

    # Random cropping
    crop_params = dict(
      offset_height=random.randint(0, height - cropped_height),
      offset_width=random.randint(0, width - cropped_width),
      target_height=cropped_height,
      target_width=cropped_width,
    )
    X = tf.image.crop_to_bounding_box(X, **crop_params)
    Y = tf.image.crop_to_bounding_box(Y, **crop_params)

    # Horizontal mirror
    if random.random() > 0.5:
      X = tf.image.flip_left_right(X)
      Y = tf.image.flip_left_right(Y)

    # Vertical mirror
    if random.random() > 0.5:
      X = tf.image.flip_up_down(X)
      Y = tf.image.flip_up_down(Y)
  return X, Y


for _ in range(10):
  display_sample(*transform_data(example_input, example_annotation))

## Creating a Keras generator

To define a Keras generator, we will use the class [`keras.utils.Sequence`](https://www.tensorflow.org/api_docs/python/tf/keras/utils/Sequence). To do this, we need two main methods: `__getitem__`, which is called at each iteration to retrieve the batch to be processed, and `__len__`, which is called to find out the total number of batches that will be supplied by the generator. We will also use `on_epoch_end` to shuffle the dataset before the next iteration.

In [ ]:
class DataGenerator(keras.utils.Sequence):
  """Generate data in batches with appropriate transforms."""

  def __init__(
    self,
    df: pandas.DataFrame,
    batch_size: int,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the generator."""
    # Votre code ici

  def __len__(self):
    """Compute the number of batches per epoch."""
    # Votre code ici

  def __getitem__(self, index):
    """Generate one batch of data."""
    # Votre code ici

  def on_epoch_end(self):
    """Update indexes after each epoch."""
    # Votre code ici

### Solution

In [ ]:
class DataGenerator(keras.utils.Sequence):
  """Generate data for Keras."""

  def __init__(
    self,
    df: pandas.DataFrame,
    batch_size: int,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the generator."""
    self.X_paths = df.Observations
    self.Y_paths = df.Annotations
    assert self.X_paths.shape[0] == self.Y_paths.shape[0]
    self.batch_size = batch_size
    self.eval = eval
    self.cropped_size = cropped_size
    if eval:
      self.indices = numpy.arange(self.X_paths.shape[0])
    else:
      self.indices = numpy.random.permutation(self.X_paths.shape[0])

  def __len__(self):
    """Compute the number of batches per epoch."""
    return int(math.ceil(self.X_paths.shape[0] / self.batch_size))

  def __getitem__(self, index):
    """Generate one batch of data."""
    # Generate indexes of the batch
    start = index * self.batch_size
    Xs = []
    Ys = []
    for i in self.indices[start : start + self.batch_size]:
      X, Y = transform_data(
        imread(self.X_paths[i]),
        imread(self.Y_paths[i]).argmax(axis=-1)[..., numpy.newaxis],
        self.eval,
        self.cropped_size,
      )
      Xs.append(X)
      Ys.append(tf.squeeze(Y))
    return tf.stack(Xs), tf.stack(Ys)

  def on_epoch_end(self):
    """Update indexes after each epoch."""
    if not self.eval:
      self.indices = numpy.arange(self.X_paths.shape[0])


generator = DataGenerator(df_train, 3, True)
X, Y = generator[0]
X.shape, Y.shape

## Creating the model

For the model, we need to go from an image of dimension (batch × height × width × 3) to an output of dimension (batch × height × width).

You can take inspiration from the architecture seen in the demonstration of the autoencoders. This time, however, we will not reduce the capacity of the network at the centre of the model.

You can also see [this Keras code example page](https://keras.io/examples/vision/oxford_pets_image_segmentation/#prepare-unet-xceptionstyle-model) for more information.

In [ ]:
def get_model(starting_depth: int = 32, steps: int = 5) -> keras.Model:
  pass  # your code here

### Solution

In [ ]:
# Implementation of the U-Net model described in the paper
# https://arxiv.org/abs/1505.04597
# You can also implement any other suitable model, see for instance this
# overview: https://arxiv.org/abs/2001.05566


def downsample(inputs):
  return keras.layers.MaxPooling2D(pool_size=(2, 2))(inputs)


def upsample(inputs):
  return keras.layers.UpSampling2D(size=(2, 2))(inputs)


def semiblock(inputs, feature_maps):
  conv = keras.layers.Conv2D(feature_maps, 3, activation="relu", padding="same")(inputs)
  batch_norm = keras.layers.BatchNormalization()(conv)
  relu = keras.layers.ReLU()(batch_norm)
  dropout = keras.layers.Dropout(0.1)(relu)
  return dropout


def block(inputs, feature_maps):
  return semiblock(semiblock(inputs, feature_maps), feature_maps)


def concat(*layers):
  return keras.layers.Concatenate()(layers)


def get_model(starting_depth: int = 32, steps: int = 5) -> keras.Model:
  inputs = keras.layers.Input((dim_input, dim_input, num_channel))
  encodings = []
  depths = [min(512, starting_depth * 2**i) for i in range(steps)]
  for depth in depths:
    encodings.append(block(downsample(encodings[-1]) if encodings else inputs, depth))
  x = encodings[-1]
  for depth, encoding in zip(depths[-2::-1], encodings[-2::-1]):
    x = block(concat(encoding, upsample(x)), depth)
  outputs = keras.layers.Conv2D(1, 1, activation="sigmoid", padding="same")(x)

  model = keras.Model(inputs=inputs, outputs=outputs)

  return model

## Training

We can now use our data generation class to train our model.

On top of accuracy, the share of correctly classified pixels, we track the IoU (intersection over union) of the forest class: the area where the prediction and the target both say "forest", divided by the area where at least one of them does. It's 1 for a perfect mask, and 0 for a model that never predicts forest, whatever its accuracy.

In [ ]:
# your code here

### Solution

In [ ]:
# Model creation
model = get_model()
model.summary()
model.compile(
  optimizer=keras.optimizers.Adam(learning_rate=1e-4),
  loss="binary_crossentropy",
  metrics=[
    "accuracy",
    keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou"),
  ],
)

batch_size = 4

# Data generator creation
train_generator = DataGenerator(df_train, batch_size, eval=False)
val_generator = DataGenerator(df_val, batch_size, eval=True)

# Training the model, evaluating it on the validation generator at the end
# of each epoch
epochs = 15
model.fit(
  train_generator,
  validation_data=val_generator,
  epochs=epochs,
  callbacks=[
    keras.callbacks.EarlyStopping(patience=5, monitor="val_loss"),
    keras.callbacks.ModelCheckpoint(
      filepath="best_model.weights.h5",
      monitor="val_accuracy",
      mode="max",
      save_weights_only=True,
      save_best_only=True,
    ),
  ],
)

## Applying to test data

Once you have learned your model sufficiently, apply it to the test data: are the predictions of good quality?

In [ ]:
# your model here

### Solution

In [ ]:
# Load best model
model.load_weights("best_model.weights.h5")

test_generator = DataGenerator(df_test, 24, eval=True)
X, Y = test_generator[2]
model.evaluate(test_generator, batch_size=4)
preds = model.predict(X)[..., 0]

print(X.shape, Y.shape, preds.shape)

for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)

## Pretrained U-Net

The previous model learns everything from scratch, encoder included. An encoder already trained on another problem knows how to extract edges, textures and shapes: this is transfer learning. [MobileNetV2](https://arxiv.org/abs/1801.04381), trained to classify ImageNet photos, will be a U-Net's encoder. We freeze it (`trainable = False`): only the decoder learns. ImageNet has no infrared images, but these visual features remain useful.

The `get_pretrained_unet` function prepares the encoder: for a 512 × 512 image, it returns in `skips` the features computed at 256, 128, 64 and 32 pixels wide, and in `x` the bottom of the "U", at 16 × 16. Complete the decoder: at each step, double the resolution (`Conv2DTranspose` with `strides=2`), concatenate the element of `skips` of the same resolution (skip connection), then apply a convolution. Finish with a last doubling, up to 512 × 512, and a layer that gives each pixel a probability of being forest.

In [ ]:
def get_pretrained_unet(
  input_height: int = dim_input, input_width: int = dim_input
) -> keras.Model:
  # Encoder: MobileNetV2 pretrained on ImageNet, without its classification
  # head. Keras warns that it loads the weights meant for 224 × 224 images: its
  # convolutions apply to any image size.
  mobilenet = keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(input_height, input_width, num_channel),
  )
  # Encoder outputs reused by the decoder, from the finest (256 × 256) to the
  # coarsest (32 × 32), then the bottom of the "U" (16 × 16)
  layer_names = [
    "block_1_expand_relu",
    "block_3_expand_relu",
    "block_6_expand_relu",
    "block_13_expand_relu",
    "block_16_project",
  ]
  encoder = keras.Model(
    mobilenet.input,
    [mobilenet.get_layer(name).output for name in layer_names],
    name="encoder",
  )
  encoder.trainable = False

  inputs = keras.Input(shape=(input_height, input_width, num_channel))
  # MobileNetV2 expects pixels between -1 and 1
  x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)
  *skips, x = encoder(x)

  # Your code here: the decoder, from x (16 × 16) to a 512 × 512 × 1 output,
  # concatenating each element of skips at its resolution
  outputs = x
  return keras.Model(inputs, outputs, name="unet")

### Solution

In [ ]:
def get_pretrained_unet(
  input_height: int = dim_input, input_width: int = dim_input
) -> keras.Model:
  # Encoder: MobileNetV2 pretrained on ImageNet, without its classification
  # head. Keras warns that it loads the weights meant for 224 × 224 images: its
  # convolutions apply to any image size.
  mobilenet = keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(input_height, input_width, num_channel),
  )
  # Encoder outputs reused by the decoder, from the finest (256 × 256) to the
  # coarsest (32 × 32), then the bottom of the "U" (16 × 16)
  layer_names = [
    "block_1_expand_relu",
    "block_3_expand_relu",
    "block_6_expand_relu",
    "block_13_expand_relu",
    "block_16_project",
  ]
  encoder = keras.Model(
    mobilenet.input,
    [mobilenet.get_layer(name).output for name in layer_names],
    name="encoder",
  )
  encoder.trainable = False

  inputs = keras.Input(shape=(input_height, input_width, num_channel))
  # MobileNetV2 expects pixels between -1 and 1
  x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)
  *skips, x = encoder(x)

  # Decoder: at each step, double the resolution and concatenate the encoder's
  # output of the same resolution (skip connection)
  for filters, skip in zip([256, 128, 64, 32], reversed(skips)):
    x = keras.layers.Conv2DTranspose(filters, 3, strides=2, padding="same")(x)
    x = keras.layers.Concatenate()([x, skip])
    x = keras.layers.Conv2D(filters, 3, padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

  # Last doubling, up to the input image's resolution
  x = keras.layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu")(x)
  outputs = keras.layers.Conv2D(1, 1, activation="sigmoid")(x)
  return keras.Model(inputs, outputs, name="unet")

### Training and comparison

Train this U-Net like the first model, with the same generators. Then compare both models on the test data: accuracy, forest IoU, and predictions on the same tiles. Which one draws the forest edges best?

In [ ]:
# your code here

### Solution

In [ ]:
unet = get_pretrained_unet()
unet.summary()
unet.compile(
  # Only the decoder, which starts from scratch, learns: a larger learning rate
  # than for the first model suits it
  optimizer=keras.optimizers.Adam(learning_rate=1e-3),
  loss="binary_crossentropy",
  metrics=[
    "accuracy",
    keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou"),
  ],
)

unet.fit(
  train_generator,
  validation_data=val_generator,
  epochs=epochs,
  callbacks=[
    keras.callbacks.EarlyStopping(patience=5, monitor="val_loss"),
    keras.callbacks.ModelCheckpoint(
      filepath="best_unet.weights.h5",
      monitor="val_accuracy",
      mode="max",
      save_weights_only=True,
      save_best_only=True,
    ),
  ],
)

In [ ]:
# Load the best model
unet.load_weights("best_unet.weights.h5")

# Both models' scores on the test tiles
results = {
  "U-Net trained from scratch": model.evaluate(test_generator, return_dict=True),
  "Pretrained U-Net (MobileNetV2)": unet.evaluate(test_generator, return_dict=True),
}
print(pandas.DataFrame(results).T[["accuracy", "iou"]].round(3))

# The same test tiles as for the first model
preds = unet.predict(X)[..., 0]

for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)